Now that you've deployed your endpoint - it's time to slam it!

In [1]:
import os
import getpass
from dotenv import load_dotenv

load_dotenv()

if not os.environ.get("FIREWORKS_API_KEY"):
    os.environ["FIREWORKS_API_KEY"] = getpass.getpass("Enter your Fireworks API key: ")

Let's try with 1 request, just to verify our endpoint is alive.

Make sure you provide your own endpoint identifier! It will look something like this:

- `accounts/fireworks/models/gpt-oss-20b` (serverless)
- Or your custom on-demand deployment identifier

In [2]:
from langchain_fireworks import ChatFireworks

# REPLACE WITH YOUR OWN ENDPOINT IDENTIFIER
model_endpoint = "accounts/fireworks/models/gpt-oss-20b"

llm = ChatFireworks(model=model_endpoint)

response = llm.invoke("How much wood could a wood chuck chuck if a wood chuck could chuck wood?")
print(response.content)

Unclosed client session
client_session: <aiohttp.client.ClientSession object at 0x11c01c910>
Unclosed client session
client_session: <aiohttp.client.ClientSession object at 0x11c01c2d0>


The classic “wood‑chuck” line is a fun tongue‑twister rather than a literal question, but people have taken it as a serious hypothesis for a good time. Here’s what we know:

| What you’re asking | The “real” answer | The humorous / folklore answer |
|--------------------|-------------------|--------------------------------|
| **Do woodchucks actually chuck wood?** | No. A **woodchuck** (scientifically *Marmota monax*, also known as the groundhog) digs burrows and moves earth, not wood. They’ll gnaw on vegetation, but they don’t “chuck” chunks of timber. | “Yes, if ever a woodchuck could figure it – it would chuck as much as it could!” |
| **If it could, how much?** | In 1988, *American Scientist* published a tongue‑in‑cheek estimate. Using the burrowing activity as a proxy for volume displaced, the study found a woodchuck could move about **13 cubic feet** of dirt per day. 13 ft³ is roughly equivalent to **700–800 pounds** of wood if the wood were as dense as something a ground‑hog wou

Now, let's SLAM IT.

In [ ]:
# Activity 1: API keys — Fireworks (required), OpenAI (for RAG + RAGAS), LangSmith (optional, for cost tracing)
if not os.environ.get("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass.getpass("OpenAI API key (for OpenAI RAG + RAGAS evaluator): ") or ""
if not os.environ.get("LANGCHAIN_API_KEY"):
    try:
        os.environ["LANGCHAIN_API_KEY"] = getpass.getpass("LangSmith API key (optional, Enter to skip): ") or ""
    except Exception:
        pass
if os.environ.get("LANGCHAIN_API_KEY"):
    os.environ["LANGCHAIN_TRACING_V2"] = "true"
os.environ.setdefault("FIREWORKS_CHAT_MODEL", model_endpoint)

Request 2 failed: {"error":{"message":"You have exceeded your rate limit for this API. Please try again later. For more information, see https://docs.fireworks.ai/guides/quotas_usage/rate-limits.","param":null,"code":"RATE_LIMIT_EXCEEDED","type":"error"},"request_id":"chatcmpl-3cf8dc96cfec475f89895cf423e3acd5"}
Request 15 failed: {"error":{"message":"You have exceeded your rate limit for this API. Please try again later. For more information, see https://docs.fireworks.ai/guides/quotas_usage/rate-limits.","param":null,"code":"RATE_LIMIT_EXCEEDED","type":"error"},"request_id":"chatcmpl-9df9457bd5b8448f9355e90d1b7f8d7c"}
Request 20 failed: {"error":{"message":"You have exceeded your rate limit for this API. Please try again later. For more information, see https://docs.fireworks.ai/guides/quotas_usage/rate-limits.","param":null,"code":"RATE_LIMIT_EXCEEDED","type":"error"},"request_id":"chatcmpl-d44d8013c7fd415198dcd0dd4483afb8"}
Request 17 failed: {"error":{"message":"You have exceeded y

In [7]:
# Activity 1: Run Fireworks + OpenAI RAG once each; collect results. (RAGAS next cell only scores these — no second graph run.)
if not os.environ.get("OPENAI_API_KEY"): os.environ["OPENAI_API_KEY"] = getpass.getpass("OpenAI API key: ") or ""
if os.environ.get("LANGCHAIN_API_KEY"): os.environ["LANGCHAIN_TRACING_V2"] = "true"
os.environ.setdefault("FIREWORKS_CHAT_MODEL", model_endpoint)  # graph uses this for generation

from app.rag import _get_rag_graph
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_openai import ChatOpenAI

EVAL_QUESTIONS = ["What should cats eat? Are they carnivores?", "How much should an adult cat eat per day?", "What vaccines do cats need?"]
# Reference answers for retrieval-quality metric (context precision)
EVAL_REFERENCES = ["Cats are obligate carnivores; they need protein from meat and should not be fed a vegetarian diet. Taurine is essential.", "Adult cats typically need 24-35 calories per pound per day. Provide fresh water.", "Core vaccines include FVRCP and rabies. Indoor cats still benefit from core vaccines; discuss with your vet."]
graph = _get_rag_graph()  # retrieve + generate; generator = Fireworks (FIREWORKS_CHAT_MODEL)
prompt = ChatPromptTemplate.from_messages([("human", "\n#CONTEXT:\n{context}\n\nQUERY:\n{query}\n\nUse the context to answer. If you don't know, say \"I don't know\".")])
openai_chain = prompt | ChatOpenAI(model="gpt-4.1-mini", temperature=0) | StrOutputParser()

os.environ["LANGCHAIN_PROJECT"] = "RAG Eval - Fireworks"
fireworks_results = []
for i, q in enumerate(EVAL_QUESTIONS):
    out = graph.invoke({"question": q})
    fireworks_results.append({"user_input": q, "retrieved_contexts": [d.page_content for d in out["context"]], "response": out["response"], "reference": EVAL_REFERENCES[i]})
print("Fireworks RAG done (graph generate step used Fireworks).")

os.environ["LANGCHAIN_PROJECT"] = "RAG Eval - OpenAI"
openai_results = []
for i, q in enumerate(EVAL_QUESTIONS):
    ctx = "\n\n".join(fireworks_results[i]["retrieved_contexts"])  # same context as Fireworks run
    openai_results.append({"user_input": q, "retrieved_contexts": fireworks_results[i]["retrieved_contexts"], "response": openai_chain.invoke({"query": q, "context": ctx}), "reference": EVAL_REFERENCES[i]})
print("OpenAI RAG done.")

Fireworks RAG done (graph generate step used Fireworks).
OpenAI RAG done.


In [9]:
# RAGAS: retrieval quality (context precision), answer faithfulness, end-to-end accuracy (answer relevancy)
import pandas as pd
from ragas import EvaluationDataset, evaluate, RunConfig
from ragas.metrics import Faithfulness, ResponseRelevancy
from ragas.llms import LangchainLLMWrapper
try:
    from ragas.metrics import ContextPrecision
except ImportError:
    from ragas.metrics.collections import ContextPrecision

ds_fw = EvaluationDataset.from_pandas(pd.DataFrame(fireworks_results).fillna(""))
ds_oa = EvaluationDataset.from_pandas(pd.DataFrame(openai_results).fillna(""))
metrics = [ContextPrecision(), Faithfulness(), ResponseRelevancy()]  # retrieval quality, faithfulness, end-to-end
eval_llm = LangchainLLMWrapper(ChatOpenAI(model="gpt-4.1-mini", temperature=0))
cfg = RunConfig(timeout=120)
fw = evaluate(dataset=ds_fw, metrics=metrics, llm=eval_llm, run_config=cfg).to_pandas().mean(numeric_only=True)
oa = evaluate(dataset=ds_oa, metrics=metrics, llm=eval_llm, run_config=cfg).to_pandas().mean(numeric_only=True)
display(pd.DataFrame({"Fireworks (gpt-oss)": fw, "OpenAI (gpt-4.1-mini)": oa}))

ModuleNotFoundError: No module named 'pandas'